# LangChain RAG Types
### 1️⃣ VectorStoreRetriever — Conversational RAG with Agent

In [ ]:
# ✅ Fixed Imports (LangChain 0.2.x style)
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.agents import initialize_agent, AgentType
from langchain.tools import StructuredTool
from langchain.memory import ConversationBufferMemory
from langchain.text_splitter import CharacterTextSplitter

# ✅ Load API keys from .env
load_dotenv(".env")
openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("❌ OPENAI_API_KEY not found! Check your .env file.")
os.environ["OPENAI_API_KEY"] = openai_key
print("✅ API Key loaded successfully")

# ─────────────────────────────────────
# 1. LLM & Embeddings
# ─────────────────────────────────────
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# ─────────────────────────────────────
# 2. Load & Split Text → Vector Store
# ─────────────────────────────────────
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()
print(f"✅ Vector store created with {len(texts)} chunks")

# ─────────────────────────────────────
# 3. Conversational RAG Chain
# ─────────────────────────────────────
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False,
    verbose=False
)

# ─────────────────────────────────────
# 4. Memory for Chat History
# ─────────────────────────────────────
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# ─────────────────────────────────────
# 5. Wrap RAG as a StructuredTool
# ─────────────────────────────────────
def rag_tool_fn(question: str) -> str:
    """Answer questions using the RAG pipeline."""
    result = rag_chain.invoke({
        "question": question,
        "chat_history": []
    })
    return result["answer"]

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# ─────────────────────────────────────
# 6. Create Agent (verbose=False → clean output)
# ─────────────────────────────────────
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=False,
    memory=memory,
    handle_parsing_errors=True
)
print("✅ Agent ready\n")

# ─────────────────────────────────────
# 7. Run Conversation
# ─────────────────────────────────────
print("1️⃣  Q: What is LangChain?")
res1 = agent.run("What is LangChain?")
print("   A:", res1)

print("\n2️⃣  Q: Who created it?")
res2 = agent.run("Who created it?")
print("   A:", res2)

print("\n3️⃣  Q: Explain LangChain again simply.")
res3 = agent.run("Explain LangChain again simply.")
print("   A:", res3)
